# Run the bridges dataset builder (Kaggle or Colab)

This notebook only **calls** the code in `bridgebuild/`; all the logic lives in the Python files.

| Step | Command | Where to run |
|---|---|---|
| 1 Setup + HF token | – | any |
| 2 Dry run (local folder, 3 cells) | `build --local-dir` | any, 2 min |
| 3 Discover regions with bridges | `discover` | any |
| 4 Build → Hub (resumable, batched) | `build` | **Kaggle CPU** (32 GB RAM, 12 h) or Colab |
| 5 Finalize: split, leakage tests, manifest, card | `finalize` | any |
| 6 Analysis + overlay samples | `stats` | any |
| 7 Export YOLO-OBB folder + train | `to-yolo` | **Kaggle GPU** (P100 / 2×T4) or Colab T4 |

**Why it does not crash RAM:** one ~4 km cell at a time, imagery read one 256 px chip at a time, chips
flushed to a Parquet shard every 500 chips, each shard uploaded with the progress file and then deleted.
If the session dies, re-run step 4: finished cells are skipped (at most one shard of work is lost).

**Kaggle:** Settings → Internet **on**; Add-ons → Secrets → `HF_TOKEN` (a *write* token).
**Colab:** 🔑 Secrets panel → `HF_TOKEN`, notebook access on.

## 1 · Setup

In [1]:
import os, sys, subprocess
ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules
BASE = "/kaggle/working" if ON_KAGGLE else ("/content" if ON_COLAB else os.getcwd())
print("Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "local", "| base:", BASE)

# OmdenaAI/HOTOSM-fAIr-Bridges is private - needs a GitHub token (same pattern as HF_TOKEN below).
# Kaggle: Add-ons -> Secrets -> GH_TOKEN (a fine-grained PAT, read-only, scoped to this repo).
GH_TOKEN = None
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    try:
        GH_TOKEN = UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        GH_TOKEN = None
elif ON_COLAB:
    from google.colab import userdata
    try:
        GH_TOKEN = userdata.get("GH_TOKEN")
    except Exception:
        GH_TOKEN = None

# Get the code: EITHER clone the shared repo OR upload bridges-hf-builder.zip
_auth = f"{GH_TOKEN}@" if GH_TOKEN else ""
REPO_URL = f"https://{_auth}github.com/OmdenaAI/HOTOSM-fAIr-Bridges.git"   # builder sits in experiments/bridges-hf-builder
REPO_DIR = f"{BASE}/HOTOSM-fAIr-Bridges"
CODE = f"{REPO_DIR}/experiments/bridges-hf-builder"
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)   # persisted session: pull latest fixes/config
elif REPO_URL and GH_TOKEN:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
elif os.path.exists(f"{BASE}/bridges-hf-builder.zip"):
    subprocess.run(["unzip", "-q", f"{BASE}/bridges-hf-builder.zip", "-d", BASE], check=True)
    CODE = f"{BASE}/bridges-hf-builder"
else:
    # Kaggle: attach the zip as a Dataset -> it appears under /kaggle/input/<name>/
    import glob
    z = glob.glob("/kaggle/input/*/bridges-hf-builder*.zip") + glob.glob("/kaggle/input/*/bridges-hf-builder")
    assert z, "set a GH_TOKEN secret, or upload bridges-hf-builder.zip"
    subprocess.run(["unzip", "-q", z[0], "-d", BASE], check=True) if z[0].endswith(".zip") else subprocess.run(["cp", "-r", z[0], BASE], check=True)
    CODE = f"{BASE}/bridges-hf-builder"
os.chdir(CODE); print("code:", os.getcwd())

Kaggle | base: /kaggle/working


Cloning into '/kaggle/working/bridges-hf-builder'...


code: /kaggle/working/bridges-hf-builder


In [2]:
!pip install -q -r requirements.txt
!python tests/test_units.py

UNIT TESTS PASSED


In [3]:
# HF token from the platform's secret store - never paste it in the notebook
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
elif ON_COLAB:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import whoami
me = whoami()["name"]; print("logged in as", me)

CONFIG = "configs/phase1_oam_france.yaml"
os.environ["BB_REPO_ID"] = f"{me}/fair-bridges-chips"     # or your org: "hotosm/fair-bridges-chips"
os.environ["PYTHONUNBUFFERED"] = "1"
print("dataset repo:", os.environ["BB_REPO_ID"])

logged in as AyaHassan
dataset repo: AyaHassan/fair-bridges-chips


## 2 · Dry run (writes to a local folder, nothing uploaded)
Checks that OAM, Overpass, IGN WFS/WMS answer from this machine. Look at the per-cell lines: bridges, images, chips, RAM.

In [ ]:
!python -m bridgebuild build --config $CONFIG --local-dir bb_dry_hub --max-cells 3
!python -m bridgebuild finalize --config $CONFIG --local-dir bb_dry_hub
!python -m bridgebuild stats --config $CONFIG --local-dir bb_dry_hub --overlays 12

In [ ]:
from IPython.display import Image as Show, display
display(Show("bb_work/reports/overlay_samples.png"))

## 3 · Discover where OAM imagery has bridges
Ranks open-licence OAM images in a search box by the number of OSM bridges inside. Copy the best ones into `regions:` of the config (step 4 only spends time where there are bridges).

In [5]:
import sys; sys.path.insert(0, ".")
from bridgebuild.discover import discover, as_regions
import yaml
cand = discover((85.20, 27.60, 85.45, 27.80), max_gsd_m=0.6, out_csv="candidates_kathmandu.csv")   # W S E N
display(cand.head(15))
print(yaml.safe_dump({"regions": as_regions(cand, "kathmandu", top=5, country="NP", global_south=True)}, sort_keys=False))

  [net] POST https://overpass-api.de/api/interpreter failed (OSError: HTTP 502), retry 1/2 in 5s
  6a4b7a536332d62593121542: 0 bridges, 0 km2, 0.01 m, CC-BY 4.0
  5fea8e72e66cea000786273a: 1 bridges, 0 km2, 0.02 m, CC-BY 4.0
  6a4b476a6332d6259311d49d: 0 bridges, 0 km2, 0.02 m, CC-BY 4.0
  6a7d3b3b782d06d7d0eb4d8d: 0 bridges, 0 km2, 0.02 m, CC-BY 4.0
  5a00bc02bac48e5b1cf744a1: 2 bridges, 1 km2, 0.04 m, CC-BY 4.0
  5fbfc070d0c51c0005eb1e02: 0 bridges, 0 km2, 0.04 m, CC-BY 4.0
  5d1eeaf25407730005b812e6: 0 bridges, 0 km2, 0.05 m, CC-BY 4.0
  6998168f4555c9ccdd1c2cef: 0 bridges, 0 km2, 0.05 m, CC-BY 4.0
  67c163ad1e8d5102b3553732: 0 bridges, 0 km2, 0.05 m, CC-BY 4.0
  67b805822a771432e43e43bd: 0 bridges, 0 km2, 0.06 m, CC-BY 4.0
  67b805822a771432e43e43c2: 0 bridges, 0 km2, 0.06 m, CC-BY 4.0
  67b865916f33238dbf8a7cb0: 0 bridges, 1 km2, 0.06 m, CC-BY 4.0
  [net] POST https://overpass-api.de/api/interpreter failed (OSError: HTTP 502), retry 2/2 in 10s
  671cb2f89954a00001688d6c: 0 bridges

,image_id,title,provider,licence,gsd_m,date,bbox,area_km2,osm_bridges,url
4,5a00bc02bac48e5b1cf744a1,Kathmandu Ricefields,WeRobotics,CC-BY 4.0,0.042810,2016-09-09,"[85.42952, 27.72362, 85.43808, 27.7284]",0.5,2,https://oin-hotosm-temp.s3.amazonaws.com/5a00b...
15,66ff858d780b5d00014e67ea,Dukuchhap_Sunakothi1,"Niroj Karmachaya,NEC",CC-BY 4.0,0.100000,2024-10-01,"[85.30788, 27.61414, 85.31388, 27.63256]",1.4,2,https://oin-hotosm-temp.s3.us-east-1.amazonaws...
1,5fea8e72e66cea000786273a,DalluV2,Raj,CC-BY 4.0,0.018200,2020-11-25,"[85.33473, 27.68971, 85.33581, 27.69086]",0.0,1,https://oin-hotosm-temp.s3.amazonaws.com/5fea8...
16,66ff865c780b5d00014e67ec,Dukuchhap_Sunakothi2,"Niroj Karmachaya,NEC",CC-BY 4.0,0.100000,2024-10-01,"[85.30878, 27.6, 85.32118, 27.61642]",2.5,1,https://oin-hotosm-temp.s3.us-east-1.amazonaws...
0,6a4b7a536332d62593121542,Chakupat/Pulchowk,Rabi Shrestha,CC-BY 4.0,0.012510,2026-03-09,"[85.32052, 27.67999, 85.3215, 27.68092]",0.0,0,https://oin-hotosm-temp.s3.us-east-1.amazonaws...
2,6a4b476a6332d6259311d49d,Lamatar Orthomosaic,Rabi Shrestha,CC-BY 4.0,0.019150,2026-03-02,"[85.39302, 27.61635, 85.39811, 27.62018]",0.2,0,https://oin-hotosm-temp.s3.us-east-1.amazonaws...
3,6a7d3b3b782d06d7d0eb4d8d,Baniyataar,Rabi Shrestha,CC-BY 4.0,0.022240,2026-07-05,"[85.31891, 27.74855, 85.32173, 27.75129]",0.1,0,https://oin-hotosm-temp.s3.us-east-1.amazonaws...
7,6998168f4555c9ccdd1c2cef,data,hari,CC-BY 4.0,0.050000,2026-02-17,"[85.32749, 27.7296, 85.32893, 27.73026]",0.0,0,https://oin-hotosm-temp.s3.us-east-1.amazonaws...
8,67c163ad1e8d5102b3553732,Task of 2024-10-01T05:58:42.459Z,Niraj Adhikari,CC-BY 4.0,0.050000,2024-05-03,"[85.32749, 27.7296, 85.32893, 27.73026]",0.0,0,https://oin-hotosm-temp.s3.us-east-1.amazonaws...
5,5fbfc070d0c51c0005eb1e02,Dallu,Raj,CC-BY 4.0,0.044658,2020-11-25,"[85.29741, 27.70864, 85.30124, 27.71257]",0.2,0,https://oin-hotosm-temp.s3.amazonaws.com/5fbfb...


regions: []



In [10]:
import yaml
from bridgebuild.discover import discover, as_regions

# Wide SEARCH boxes (discover finds real images inside them; as_regions keeps only the good ones).
# Picked for known OAM/HOT mapping activity (disaster response campaigns tend to have dense OAM coverage).
CANDIDATES = {
    "beira":          ((34.75, -19.95, 34.95, -19.75), "MZ"),   # Cyclone Idai response
    "port_au_prince": ((-72.45, 18.48, -72.25, 18.62), "HT"),   # extensive OAM archive since 2010
    "tacloban":       ((124.95, 11.17, 125.10, 11.30), "PH"),   # Typhoon Haiyan response
    "monrovia":       ((-10.85, 6.25, -10.70, 6.40), "LR"),
    "freetown":       ((-13.30, 8.40, -13.15, 8.55), "SL"),     # 2017 mudslide response
    "blantyre":       ((35.00, -15.85, 35.10, -15.75), "MW"),   # Idai-affected
}

all_regions = []
for name, (bbox, country) in CANDIDATES.items():
    print(f"--- {name} ---")
    cand = discover(bbox, max_gsd_m=0.6, out_csv=f"candidates_{name}.csv")
    regions = as_regions(cand, name, top=5, min_bridges=5, country=country, global_south=True)
    print(f"  {len(regions)} usable region(s)")
    all_regions += regions

print(f"\n{len(all_regions)} total candidate regions found\n")
print(yaml.safe_dump({"regions": all_regions}, sort_keys=False))


--- beira ---
  693968138658276b90f7e762: 16 bridges, 2 km2, 0.05 m, CC-BY 4.0
  69205c0635291abb7565a7f5: 5 bridges, 1 km2, 0.05 m, CC-BY 4.0
  693c213f5031d65a77d261cf: 15 bridges, 3 km2, 0.05 m, CC-BY 4.0
  [net] POST https://overpass-api.de/api/interpreter failed (OSError: HTTP 502), retry 1/2 in 5s
  693c217f5031d65a77d2625a: 10 bridges, 3 km2, 0.05 m, CC-BY 4.0
  693c18f45031d65a77d24e1f: 10 bridges, 3 km2, 0.05 m, CC-BY 4.0
  [net] POST https://overpass-api.de/api/interpreter failed (OSError: HTTP 502), retry 1/2 in 5s
  693c6ed15031d65a77d2a3b8: 24 bridges, 3 km2, 0.05 m, CC-BY 4.0
  693968458658276b90f7e7f3: 24 bridges, 3 km2, 0.05 m, CC-BY 4.0
  693d974985300c176da58c53: 9 bridges, 1 km2, 0.05 m, CC-BY 4.0
  693cc67ff9bbc95e9c58538a: 9 bridges, 1 km2, 0.05 m, CC-BY 4.0
  5 usable region(s)
--- port_au_prince ---
  [net] POST https://overpass-api.de/api/interpreter failed (OSError: HTTP 502), retry 1/2 in 5s
  69bbb04457bbd74f1e80e577: 1 bridges, 1 km2, 0.03 m, CC-BY 4.0
  69b

## 4 · Build → Hugging Face (resumable)
Re-run this cell after any disconnect; it continues where it stopped. `--max-hours 11` stops cleanly before Kaggle's 12 h limit.
Use `--only oam_osm` / `--only france` to build one source per session (e.g. two Kaggle sessions in parallel would clash on the same repo — run them one after another).

In [22]:
!python -m bridgebuild build --only oam_osm --config $CONFIG --max-hours 12

state.json: 100%|██████████████████████████| 3.70k/3.70k [00:00<00:00, 8.23MB/s]
[build] target: https://huggingface.co/datasets/AyaHassan/fair-bridges-chips | already done: 36 cells, 2008 chips
[build] done for now: 36 cells, 2008 chips (1034 with bridges, 1316 boxes), 0 failed cells, 5 shards


In [25]:
!git -C $CODE pull --ff-only   # or just re-run cell 2 if it already pulls
!python -m bridgebuild build --config $CONFIG --only oam_osm --retry-failed
!python -m bridgebuild finalize --config $CONFIG
!python -m bridgebuild stats --config $CONFIG --push


Already up to date.
[build] target: https://huggingface.co/datasets/AyaHassan/fair-bridges-chips | already done: 36 cells, 2008 chips
[build] done for now: 36 cells, 2008 chips (1034 with bridges, 1316 boxes), 0 failed cells, 5 shards
state.json: 100%|██████████████████████████| 3.70k/3.70k [00:00<00:00, 4.95MB/s]
Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  ...rk/finalize/split.parquet: 100%|██████████████| 20.5kB / 20.5kB            

Processing Files (1 / 1)      : 100%|██████████████| 20.5kB / 20.5kB,   ???B/s  

  ...rk/finalize/split.parquet: 100%|██████████████| 20.5kB / 20.5kB            

  ...rk/finalize/split.parquet: 100%|██████████████| 20.5kB / 20.5kB            

  ...rk/finalize/split.parquet: 100%|██████████████| 20.5kB / 20.5kB            

Processing Files (1 / 1)      : 100%|██████████████| 20.5kB / 20.5kB, 1.98kB/s  
New Data Upload               

In [ ]:
# Progress (from the Hub, no images downloaded)
import json
from huggingface_hub import hf_hub_download
st = json.load(open(hf_hub_download(os.environ["BB_REPO_ID"], "state/state.json", repo_type="dataset", force_download=True)))
print(st["totals"], "| failed cells:", len(st["failed_cells"]))
list(st["failed_cells"].items())[:10]

In [ ]:
# Optional: retry cells that failed because a service was down
!python -m bridgebuild build --config $CONFIG --retry-failed --max-hours 2

## 5 · Finalize: split + leakage tests + manifest + dataset card + record
Reads only the small index files. Fails (exit code 2) if any leakage test fails.

In [ ]:
!python -m bridgebuild finalize --config $CONFIG

## 6 · Analysis + overlay samples (pushed to `reports/` on the Hub)

In [ ]:
!python -m bridgebuild stats --config $CONFIG --push --overlays 24
import pandas as pd, glob
for f in ["chips_by_split", "bridges_by_source", "bridge_coverage", "licences"]:
    p = f"bb_work/reports/{f}.csv"
    if os.path.exists(p): print(f); display(pd.read_csv(p))
display(Show("bb_work/reports/overview.png")); display(Show("bb_work/reports/overlay_samples.png"))

In [ ]:
# Your own analysis: stream the dataset (no full download)
from datasets import load_dataset
ds = load_dataset(os.environ["BB_REPO_ID"], split="train", streaming=True)
ex = next(iter(ds)); print({k: v for k, v in ex.items() if k != "image"}); ex["image"]

## 7 · Export YOLO-OBB and train (GPU session)
Streams one shard at a time to `images/` + `labels/` + `data.yaml`. `--neg-keep 0.5` keeps half of the negative chips.

In [ ]:
OUT = f"{BASE}/yolo_bridges"
!python -m bridgebuild to-yolo --config $CONFIG --out $OUT --neg-keep 1.0
!du -sh $OUT; cat $OUT/data.yaml

In [ ]:
# Training with the doc 04 §6 settings. Needs a GPU session.
!pip install -q ultralytics
from ultralytics import YOLO
model = YOLO("yolo26s-obb.pt")          # DOTA weights; nc is reset to 1 from data.yaml
model.train(data=f"{OUT}/data.yaml", imgsz=256, epochs=100, batch=64, patience=20, close_mosaic=10,
            degrees=180, flipud=0.5, fliplr=0.5, scale=0.3, hsv_h=0.02, hsv_v=0.3, project=f"{BASE}/runs", name="phase1a")

In [ ]:
# Push trained weights to the HF Hub instead of leaving them only in /kaggle/working (wiped on session end)
from huggingface_hub import HfApi

YOLO_REPO_ID = f"{me}/fair-bridges-yolo"   # or your org: "hotosm/fair-bridges-yolo"
RUN_DIR = f"{BASE}/runs/phase1a"

api = HfApi()
api.create_repo(YOLO_REPO_ID, repo_type="model", private=True, exist_ok=True)
api.upload_folder(repo_id=YOLO_REPO_ID, repo_type="model", folder_path=f"{RUN_DIR}/weights",
                   path_in_repo="weights", commit_message="phase1a: best + last weights")
api.upload_file(repo_id=YOLO_REPO_ID, repo_type="model", path_or_fileobj=f"{RUN_DIR}/args.yaml",
                 path_in_repo="args.yaml")
print(f"weights pushed to https://huggingface.co/{YOLO_REPO_ID}")


In [ ]:
# Visualize test predictions vs ground truth (run after cell 21's model.train())
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Polygon as MplPolygon
from PIL import Image

TEST_IMG_DIR = Path(OUT) / "images" / "test"
TEST_LBL_DIR = Path(OUT) / "labels" / "test"
N_SAMPLES = 12
CONF_THRES = 0.25   # tune to taste; mAP/precision/recall above were computed at ultralytics' own sweep, not this value

images = sorted(TEST_IMG_DIR.glob("*.png"))
if not images:
    print("no test images found under", TEST_IMG_DIR)
else:
    sample = random.Random(0).sample(images, min(N_SAMPLES, len(images)))
    results = model.predict(source=[str(p) for p in sample], conf=CONF_THRES, verbose=False)

    cols = 4
    rows = -(-len(sample) // cols)
    fig, axs = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 3.2), squeeze=False)
    for ax in axs.flat:
        ax.axis("off")

    for ax, img_path, r in zip(axs.flat, sample, results):
        im = np.array(Image.open(img_path).convert("RGB"))
        h, w = im.shape[:2]
        ax.imshow(im)

        lbl_path = TEST_LBL_DIR / f"{img_path.stem}.txt"
        gt_lines = lbl_path.read_text().splitlines() if lbl_path.exists() else []
        for line in gt_lines:
            vals = list(map(float, line.split()[1:]))
            pts = np.array(vals).reshape(4, 2) * [w, h]
            ax.add_patch(MplPolygon(pts, closed=True, fill=False, edgecolor="lime", linewidth=1.5))

        n_pred = len(r.obb) if r.obb is not None else 0
        if n_pred:
            for pts, conf in zip(r.obb.xyxyxyxy.cpu().numpy(), r.obb.conf.cpu().numpy()):
                ax.add_patch(MplPolygon(pts, closed=True, fill=False, edgecolor="red", linewidth=1.5))
                ax.text(pts[:, 0].min(), pts[:, 1].min() - 2, f"{conf:.2f}", color="red", fontsize=7,
                        bbox=dict(facecolor="black", alpha=0.4, pad=0.5, linewidth=0))

        ax.set_title(f"{img_path.stem}\nGT {len(gt_lines)} | pred {n_pred}", fontsize=7)

    fig.suptitle("Test predictions (red) vs ground truth (green)", y=1.0)
    plt.tight_layout()
    out_png = f"{BASE}/runs/phase1a/test_predictions.png"
    plt.savefig(out_png, dpi=130, bbox_inches="tight")
    plt.show()
    print("saved:", out_png)


In [ ]:
results_low = model.predict(source=[str(p) for p in sample], conf=0.01, verbose=False)
for img_path, r in zip(sample, results_low):
    confs = r.obb.conf.cpu().numpy().tolist() if r.obb is not None else []
    print(img_path.stem, "max conf:", max(confs) if confs else None)


In [ ]:
metrics_test = model.val(data=f"{OUT}/data.yaml", split="test")
print(metrics_test.results_dict)
